In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install dspy

In [ ]:
! pip install litellm[proxy]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.9/187.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.9/412.9 kB 26.3 MB/s eta 0:00:00
   ━━━━━

In [ ]:
key=''

In [ ]:
import dspy
lm = dspy.LM('openai/gpt-4o', api_key=key)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal, List
import dspy

class ClassifyDiabetesStigma(dspy.Signature):
    """
    Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    category: Literal['yes stigma', 'no stigma'] = dspy.OutputField(
        desc="Does the post contain diabetes-related stigma?"
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")




class ClassifyDiabetesStigmaType(dspy.Signature):
    """
    Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
    identify the types of stigma present.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    stigma_type: List[Literal[
        'experienced stigma',
        'perceived stigma',
        'anticipated stigma',
        'internalized stigma',
        'intersectional stigma'
    ]] = dspy.OutputField(
        desc="List of stigma types present in the post."
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")



In [ ]:
classify_stigma = dspy.Predict(ClassifyDiabetesStigma)
classify_stigma(title = 'I don\'t know what t do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='no stigma',
    confidence=0.85
)

In [ ]:
import json
with open('/content/drive/MyDrive/stigma_project/final experiment results/train_data.json', 'r') as f:
  train_data = json.loads(f.read())
  f.close()

with open('/content/drive/MyDrive/stigma_project/final experiment results/val_data.json', 'r') as f:
  val_data = json.loads(f.read())
  f.close()

In [ ]:
f_train_data = [d for d in train_data if d['label'] == 'yes stigma']
f_val_data = [d for d in val_data if d['label'] == 'yes stigma']

In [ ]:
l_trainset = []

for post_obj in f_train_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], category=post_obj['label']).with_inputs("title", "post")
    l_trainset.append(example)


l_valset = []

for post_obj in f_val_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], category=post_obj['label']).with_inputs("title", "post")
    l_valset.append(example)

In [ ]:
prompt_gen_lm = dspy.LM('openai/gpt-4o-mini', api_key=key)

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def accuracy_metric(example, prediction, trace=None):
    return prediction.category == example.category

In [ ]:
def combined_f1_metric(example, prediction, trace=None):
    # ----- CATEGORY (single-label classification) -----
    true_cat = example['category']
    pred_cat = prediction.category

    category_f1 = f1_score([true_cat], [pred_cat], average='macro', zero_division=1)

    # ----- STIGMA TYPE (multi-label classification) -----
    true_stigma = example.get('stigma_type', [])
    pred_stigma = prediction.stigma_type if prediction.stigma_type is not None else []

    # Align possible labels
    all_labels = ['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

    # Convert to binary format for multi-label F1
    def to_multilabel(y, labels):
        return [1 if label in y else 0 for label in labels]

    y_true_bin = [to_multilabel(true_stigma, all_labels)]
    y_pred_bin = [to_multilabel(pred_stigma, all_labels)]

    stigma_f1 = f1_score(y_true_bin, y_pred_bin, average='micro', zero_division=1)

    # ----- COMBINED SCORE -----
    combined_score = 0.5 * category_f1 + 0.5 * stigma_f1
    return combined_score


# Label detection optimization

# Bootstap few-shot

In [ ]:
from dspy.teleprompt import BootstrapFewShot

fewshot_optimizer = BootstrapFewShot(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16)

optimized_classify = fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset)

 11%|█         | 11/100 [00:00<00:05, 15.01it/s]

Bootstrapped 8 full traces after 11 examples for up to 1 rounds, amounting to 11 attempts.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_bootstrap_fewshot_optimized.json')

In [ ]:
del optimized_classify

# Bootstrap fewshot random search

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3, num_threads=3)

optimized_classify = fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset, valset=l_valset)

Going to sample between 1 and 8 traces per predictor.
Will attempt to bootstrap 3 candidate sets.
Average Metric: 3.00 / 10 (30.0%): 100%|██████████| 10/10 [00:05<00:00,  1.77it/s]

2025/08/08 11:14:45 INFO dspy.evaluate.evaluate: Average Metric: 3 / 10 (30.0%)



New best score: 30.0 for seed -3
Scores so far: [30.0]
Best score so far: 30.0
Average Metric: 6.00 / 10 (60.0%): 100%|██████████| 10/10 [00:03<00:00,  2.51it/s]

2025/08/08 11:14:49 INFO dspy.evaluate.evaluate: Average Metric: 6 / 10 (60.0%)



New best score: 60.0 for seed -2
Scores so far: [30.0, 60.0]
Best score so far: 60.0


...


Scores so far: [30.0, 60.0, 80.0, 70.0, 80.0]
Best score so far: 80.0


  2%|▏         | 2/100 [00:01<01:32,  1.06it/s]


Bootstrapped 1 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:03<00:00,  2.87it/s]

2025/08/08 11:15:26 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



Scores so far: [30.0, 60.0, 80.0, 70.0, 80.0, 80.0]
Best score so far: 80.0
6 candidate programs found.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_bootstrap_fewshot_random_search_optimized.json')

In [ ]:
del optimized_classify

# MIPROV2

In [ ]:
tp = dspy.MIPROv2(metric=accuracy_metric,
                  auto = None,
                  prompt_model=prompt_gen_lm,
                  num_candidates=3,
                  init_temperature=0,
                  task_model=lm)

optimized_classify = tp.compile(classify_stigma, trainset=l_trainset, valset=l_valset,
                  max_labeled_demos=16,
                  max_bootstrapped_demos=8,
                  minibatch_size=10,
                  num_trials=6,
                  minibatch_full_eval_steps=10,
                  requires_permission_to_run=False)

2025/08/08 11:17:01 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/08 11:17:01 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/08 11:17:01 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 11%|█         | 11/100 [00:00<00:01, 84.99it/s]
2025/08/08 11:17:01 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/08/08 11:17:01 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Bootstrapped 8 full traces after 11 examples for up to 1 rounds, amounting to 11 attempts.
Error getting source code: unhashable type: 'dict'.

Running without program aware proposer.


2025/08/08 11:17:56 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=3 instructions...

2025/08/08 11:18:01 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/08/08 11:18:01 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.

2025/08/08 11:18:01 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Classify the following social media post (including title and body) as containing stigma toward people with diabetes or not.

2025/08/08 11:18:01 INFO dspy.teleprompt.mipro_optimizer_v2: 2: You are a social media analyst tasked with identifying and classifying posts that may contain stigma toward individuals with diabetes. Read the provided social media post, which includes a title and body, and determine whether it expresses stigma against people with diabetes. Provide your classification as "yes stigma" or "no stigma" based on the content of the post.

2025/08

...


Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:00<00:00, 425.60it/s]

2025/08/08 11:18:09 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 80.0 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [30.0, 80.0, 80.0, 60.0, 70.0, 80.0, 80.0]
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 30.0
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 8 / 8 - Full Evaluation =====
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 80.0) from minibatch trials...



Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:00<00:00, 742.70it/s]

2025/08/08 11:18:09 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 80.0
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [30.0, 80.0, 80.0, 60.0, 70.0, 80.0, 80.0, 80.0]


2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/08/08 11:18:09 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 80.0!


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_MIPROV2_optimized.json')

In [ ]:
del optimized_classify

# COPRO

In [ ]:
import litellm
litellm.drop_params = True

In [ ]:
import litellm
litellm.drop_params = True

from dspy.teleprompt import COPRO

eval_kwargs = dict(num_threads=16, display_progress=True, display_table=0)

copro_teleprompter = COPRO(prompt_model=prompt_gen_lm, metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, breadth=5, depth=3, init_temperature=0, verbose=False)

optimized_classify = copro_teleprompter.compile(classify_stigma, trainset=l_trainset, eval_kwargs=eval_kwargs)

2025/08/08 11:19:16 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/3.
2025/08/08 11:19:16 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #1/5 for Predictor 1 of 1.






[2025-08-08T11:19:16.763817]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2025/08/08 11:19:21 INFO dspy.evaluate.evaluate: Average Metric: 17 / 100 (17.0%)
2025/08/08 11:19:22 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #2/5 for Predictor 1 of 1.







[2025-08-08T11:19:16.763817]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be

2025/08/08 11:19:27 INFO dspy.evaluate.evaluate: Average Metric: 25 / 100 (25.0%)
2025/08/08 11:19:27 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #3/5 for Predictor 1 of 1.


...






[2025-08-08T11:20:19.568459]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


2025/08/08 11:20:41 INFO dspy.teleprompt.copro_optimizer: At Depth 3/3, Evaluating Prompt Candidate #5/5 for Predictor 1 of 1.


Average Metric: 22.00 / 100 (22.0%): 100%|██████████| 100/100 [00:05<00:00, 17.82it/s]

2025/08/08 11:20:47 INFO dspy.evaluate.evaluate: Average Metric: 22 / 100 (22.0%)







[2025-08-08T11:20:19.568459]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_COPRO_optimized.json')

In [ ]:
del optimized_classify

# Bootstrap few-shot with optuna

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithOptuna

fewshot_optuna_optimizer = BootstrapFewShotWithOptuna(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3)

optimized_classify = fewshot_optuna_optimizer.compile(student=classify_stigma, trainset=l_trainset, valset=l_valset, max_demos = 8)

Going to sample between 1 and 8 traces per predictor.
Will attempt to train 3 candidate sets.


 11%|█         | 11/100 [00:00<00:00, 189.75it/s]


Bootstrapped 8 full traces after 11 examples for up to 1 rounds, amounting to 11 attempts.
Average Metric: 3.00 / 10 (30.0%): 100%|██████████| 10/10 [00:04<00:00,  2.19it/s]

2025/08/08 11:21:32 INFO dspy.evaluate.evaluate: Average Metric: 3 / 10 (30.0%)



Average Metric: 3.00 / 10 (30.0%): 100%|██████████| 10/10 [00:01<00:00,  5.60it/s]

2025/08/08 11:21:34 INFO dspy.evaluate.evaluate: Average Metric: 3 / 10 (30.0%)



Average Metric: 3.00 / 10 (30.0%): 100%|██████████| 10/10 [00:00<00:00, 923.12it/s]

2025/08/08 11:21:34 INFO dspy.evaluate.evaluate: Average Metric: 3 / 10 (30.0%)



Best score: 30.0
Best program: Predict(ClassifyDiabetesStigma(title, post -> category, confidence
    instructions='Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.'
    title = Field(annotation=str required=True json_schema_extra={'desc': 'The title of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Title:'})
    post = Field(annotation=str required=True json_schema_extra={'desc': 'The main text or content of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    category = Field(annotation=Literal['yes stigma', 'no stigma'] required=True json_schema_extra={'desc': 'Does the post contain diabetes-related stigma?', '__dspy_field_type': 'output', 'prefix': 'Category:'})
    confidence = Field(annotation=float required=True json_schema_extra={'desc': "Model's confidence score.", '__dspy_field_type': 'output', 'prefix': 'Confidence:'})
))


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_bootstrap_fewshot_optuna_optimized.json')

In [ ]:
del optimized_classify

# Labled KNN fewshot

In [ ]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=8)
optimized_classify = labeled_fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset)

In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy optimization - pos samples/GPT/GPT_labeled_fewshot_optimized.json')

In [ ]:
del optimized_classify

# Type detection optimization

In [ ]:
class ClassifyDiabetesStigmaType(dspy.Signature):
    """
    Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
    identify the types of stigma present.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    stigma_type: List[Literal[
        'experienced stigma',
        'perceived stigma',
        'anticipated stigma',
        'internalized stigma',
        'intersectional stigma'
    ]] = dspy.OutputField(
        desc="List of stigma types present in the post."
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")



In [ ]:
type_detector = dspy.Predict(ClassifyDiabetesStigmaType)
type_detector(title = 'I don\'t know what t do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    stigma_type=['internalized stigma', 'experienced stigma'],
    confidence=0.85
)

In [ ]:
t_trainset = []

for post_obj in f_train_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], stigma_type=post_obj['types']).with_inputs("title", "post")
    t_trainset.append(example)


t_valset = []

for post_obj in f_val_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], stigma_type=post_obj['types']).with_inputs("title", "post")
    t_valset.append(example)

In [ ]:
def jaccard_metric(example, prediction, trace=None):
    set_p = set(prediction.stigma_type)
    set_e = set(example.stigma_type)
    return len(set_p.intersection(set_e)) / len(set_p.union(set_e))

# Bootstrap fewshot

In [ ]:
from dspy.teleprompt import BootstrapFewShot

fewshot_optimizer = BootstrapFewShot(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16)

optimized_classify = fewshot_optimizer.compile(student = type_detector, trainset=t_trainset)

  9%|▉         | 9/100 [00:04<00:46,  1.95it/s]

Bootstrapped 8 full traces after 9 examples for up to 1 rounds, amounting to 9 attempts.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_bootstrap_fewshot_optimized.json')

In [ ]:
del optimized_classify

# Bootstrap fewshot random search

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3, num_threads=3)

optimized_classify = fewshot_optimizer.compile(student = type_detector, trainset=t_trainset, valset=t_valset)

Going to sample between 1 and 8 traces per predictor.
Will attempt to bootstrap 3 candidate sets.
Average Metric: 5.00 / 10 (50.0%): 100%|██████████| 10/10 [00:03<00:00,  2.96it/s]

2025/08/08 11:38:18 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 10 (50.0%)



New best score: 50.0 for seed -3
Scores so far: [50.0]
Best score so far: 50.0
Average Metric: 7.00 / 10 (70.0%): 100%|██████████| 10/10 [00:03<00:00,  3.12it/s]

2025/08/08 11:38:21 INFO dspy.evaluate.evaluate: Average Metric: 7.0 / 10 (70.0%)



New best score: 70.0 for seed -2
Scores so far: [50.0, 70.0]
Best score so far: 70.0


...


Scores so far: [50.0, 70.0, 71.67, 71.67, 71.67]
Best score so far: 71.67


  1%|          | 1/100 [00:01<02:15,  1.37s/it]


Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Average Metric: 6.17 / 10 (61.7%): 100%|██████████| 10/10 [00:03<00:00,  2.81it/s]

2025/08/08 11:38:49 INFO dspy.evaluate.evaluate: Average Metric: 6.166666666666666 / 10 (61.7%)



Scores so far: [50.0, 70.0, 71.67, 71.67, 71.67, 61.67]
Best score so far: 71.67
6 candidate programs found.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_bootstrap_fewshot_random_search_optimized.json')

In [ ]:
del optimized_classify

# MIPROV2

In [ ]:
tp = dspy.MIPROv2(metric=jaccard_metric,
                  auto = None,
                  prompt_model=prompt_gen_lm,
                  num_candidates=3,
                  init_temperature=0,
                  task_model=lm)

optimized_classify = tp.compile(type_detector, trainset=t_trainset, valset=t_valset,
                  max_labeled_demos=16,
                  max_bootstrapped_demos=8,
                  minibatch_size=10,
                  num_trials=6,
                  minibatch_full_eval_steps=10,
                  requires_permission_to_run=False)

2025/08/08 11:40:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/08 11:40:36 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/08 11:40:36 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


  9%|▉         | 9/100 [00:00<00:00, 159.11it/s]
2025/08/08 11:40:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/08/08 11:40:36 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Bootstrapped 8 full traces after 9 examples for up to 1 rounds, amounting to 9 attempts.
Error getting source code: unhashable type: 'dict'.

Running without program aware proposer.


2025/08/08 11:41:26 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=3 instructions...

2025/08/08 11:41:31 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/08/08 11:41:31 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
identify the types of stigma present.

2025/08/08 11:41:31 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Analyze the provided social media post, including its title and body, and identify the types of stigma related to diabetes that are present in the text.

2025/08/08 11:41:31 INFO dspy.teleprompt.mipro_optimizer_v2: 2: You are a mental health professional specializing in chronic illness. Given a social media post (title and body) that expresses feelings of guilt, anxiety, or frustration related to managing diabetes, identify and categorize the types of stigma present in the post.

2025/08/08 11:41:31 INFO dspy.telepromp

...

2025/08/08 11:41:41 INFO dspy.evaluate.evaluate: Average Metric: 6.666666666666666 / 10 (66.7%)
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 66.67 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [50.0, 65.0, 71.67, 70.0, 65.0, 66.67]
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 50.0
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 7 / 8 - Minibatch ==



Average Metric: 6.67 / 10 (66.7%): 100%|██████████| 10/10 [00:00<00:00, 1062.17it/s]

2025/08/08 11:41:41 INFO dspy.evaluate.evaluate: Average Metric: 6.666666666666666 / 10 (66.7%)
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 66.67 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [50.0, 65.0, 71.67, 70.0, 65.0, 66.67, 66.67]
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 50.0
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 8 / 8 - Full Evaluation =====
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 71.67) from minibatch trials...



Average Metric: 7.17 / 10 (71.7%): 100%|██████████| 10/10 [00:00<00:00, 883.40it/s]

2025/08/08 11:41:41 INFO dspy.evaluate.evaluate: Average Metric: 7.166666666666666 / 10 (71.7%)
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 71.67
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [50.0, 65.0, 71.67, 70.0, 65.0, 66.67, 66.67, 71.67]
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 71.67
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/08/08 11:41:41 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 71.67!


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_MIPROV2_optimized.json')

In [ ]:
del optimized_classify

# COPRO

In [ ]:
import litellm
litellm.drop_params = True

from dspy.teleprompt import COPRO

eval_kwargs = dict(num_threads=16, display_progress=True, display_table=0)

copro_teleprompter = COPRO(prompt_model=prompt_gen_lm, metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, breadth=5, depth=3, init_temperature=0, verbose=False)

optimized_classify = copro_teleprompter.compile(type_detector, trainset=t_trainset, eval_kwargs=eval_kwargs)

2025/08/08 11:43:12 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/3.
2025/08/08 11:43:12 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #1/5 for Predictor 1 of 1.






[2025-08-08T11:43:12.429820]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2025/08/08 11:43:20 INFO dspy.evaluate.evaluate: Average Metric: 57.949999999999996 / 100 (58.0%)
2025/08/08 11:43:20 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #2/5 for Predictor 1 of 1.







[2025-08-08T11:43:12.429820]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be

2025/08/08 11:43:29 INFO dspy.evaluate.evaluate: Average Metric: 54.45000000000001 / 100 (54.5%)
2025/08/08 11:43:29 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #3/5 for Predictor 1 of 1.


...






[2025-08-08T11:44:37.804246]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


2025/08/08 11:45:08 INFO dspy.teleprompt.copro_optimizer: At Depth 3/3, Evaluating Prompt Candidate #5/5 for Predictor 1 of 1.


Average Metric: 53.65 / 100 (53.7%): 100%|██████████| 100/100 [00:07<00:00, 13.09it/s]

2025/08/08 11:45:15 INFO dspy.evaluate.evaluate: Average Metric: 53.65000000000002 / 100 (53.7%)







[2025-08-08T11:44:37.804246]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_COPRO_optimized.json')

In [ ]:
del optimized_classify

# Bootstrap fewshot with optuna

In [ ]:
from dspy.teleprompt import BootstrapFewShotWithOptuna

fewshot_optuna_optimizer = BootstrapFewShotWithOptuna(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3)

optimized_classify = fewshot_optuna_optimizer.compile(student=type_detector, trainset=t_trainset, valset=t_valset, max_demos = 8)

Going to sample between 1 and 8 traces per predictor.
Will attempt to train 3 candidate sets.


  9%|▉         | 9/100 [00:00<00:00, 93.74it/s]

Bootstrapped 8 full traces after 9 examples for up to 1 rounds, amounting to 9 attempts.


Average Metric: 5.67 / 10 (56.7%): 100%|██████████| 10/10 [00:01<00:00,  7.05it/s]

2025/08/08 11:46:00 INFO dspy.evaluate.evaluate: Average Metric: 5.666666666666666 / 10 (56.7%)



Average Metric: 5.17 / 10 (51.7%): 100%|██████████| 10/10 [00:01<00:00,  6.23it/s]

2025/08/08 11:46:02 INFO dspy.evaluate.evaluate: Average Metric: 5.166666666666666 / 10 (51.7%)



Average Metric: 5.00 / 10 (50.0%): 100%|██████████| 10/10 [00:01<00:00,  6.00it/s]

2025/08/08 11:46:04 INFO dspy.evaluate.evaluate: Average Metric: 5.0 / 10 (50.0%)



Best score: 56.67
Best program: Predict(ClassifyDiabetesStigmaType(title, post -> stigma_type, confidence
    instructions='Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,\nidentify the types of stigma present.'
    title = Field(annotation=str required=True json_schema_extra={'desc': 'The title of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Title:'})
    post = Field(annotation=str required=True json_schema_extra={'desc': 'The main text or content of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    stigma_type = Field(annotation=List[Literal['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']] required=True json_schema_extra={'desc': 'List of stigma types present in the post.', '__dspy_field_type': 'output', 'prefix': 'Stigma Type:'})
    confidence = Field(annotation=float required=True json_schema_extra={'

In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_bootstrap_fewshot_optuna_optimized.json')

In [ ]:
del optimized_classify

# Labeled KNN fewshot

In [ ]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=8)
optimized_classify = labeled_fewshot_optimizer.compile(student = type_detector, trainset=t_trainset)

In [ ]:
optimized_classify.save('/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_labeled_fewshot_optimized.json')

In [ ]:
del optimized_classify

# Evaluation on test data (label)

In [ ]:
import json
from tqdm import trange

# models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
# models = ['labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
models = ['COT']

for m in models:
    classify = None
    if m == 'baseline':
        classify = dspy.Predict(ClassifyDiabetesStigma)
    elif m == 'COT':
        classify = dspy.ChainOfThought(ClassifyDiabetesStigma)
    else:
        direc = f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/GPT/GPT_{m}_optimized.json'

        classify = dspy.Predict(ClassifyDiabetesStigma)
        classify.load(direc)

    print(f'Model {m} loaded!')

    with open('/content/drive/MyDrive/stigma_project/final experiment results/test_data.json', 'r') as f:
        test_data = json.loads(f.read())
        f.close()

    preds = []
    for i in trange(len(test_data)):
        pred = classify(title=test_data[i]['title'], post=test_data[i]['post text'])

        test_data[i]['pred_label'] = pred.category
        # test_data[i]['pred_types'] = pred.stigma_type
        test_data[i]['confidence'] = pred.confidence
        preds.append(test_data[i])

    with open(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/GPT/GPT_{m}_test_data_pred.json', 'w') as f:
        f.write(json.dumps(preds))
        f.close()

    print(f'Results for GPT {m} saved!')

Model COT loaded!


100%|██████████| 302/302 [12:25<00:00,  2.47s/it]

Results for GPT COT saved!


# Evaluation on test data (types)

In [ ]:
import json
from tqdm import trange

# models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
models = ['COT']

for m in models:
    type_detector = None
    if m == 'baseline':
        type_detector = dspy.Predict(ClassifyDiabetesStigmaType)
    elif m == 'COT':
        type_detector = dspy.ChainOfThought(ClassifyDiabetesStigmaType)
    else:
        direc = f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_{m}_optimized.json'

        type_detector = dspy.Predict(ClassifyDiabetesStigmaType)
        type_detector.load(direc)

    print(f'Model {m} loaded!')

    with open('/content/drive/MyDrive/stigma_project/final experiment results/test_data.json', 'r') as f:
        test_data = json.loads(f.read())
        f.close()

    preds = []
    for i in trange(len(test_data)):
        pred = type_detector(title=test_data[i]['title'], post=test_data[i]['post text'])

        # test_data[i]['pred_label'] = pred.category
        test_data[i]['pred_types'] = pred.stigma_type
        test_data[i]['confidence'] = pred.confidence
        preds.append(test_data[i])

    with open(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_{m}_test_data_pred.json', 'w') as f:
        f.write(json.dumps(preds))
        f.close()

    print(f'Results for GPT {m} saved!')

Model COT loaded!


 10%|▉         | 29/302 [02:11<20:39,  4.54s/it]


RuntimeError: Both structured output format and JSON mode failed. Please choose a model that supports `response_format` argument. Original error: litellm.RateLimitError: RateLimitError: OpenAIException - You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.

# Get report files from preds.json



In [ ]:
%%capture
!pip install imbalanced-learn

In [3]:
import json
from tqdm import tqdm, trange
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score
# from imblearn.metrics import make_index_balanced_accuracy

In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)


def jaccard_score(df_list):
  # assert len(true_list) == len(pred_list)

  scores = []
  for row in df_list.itertuples(index=False):
    if len(row.types) == 0 and len(row.pred_types) == 0:
      continue
    else:
      true_set = set(row.types)
      pred_set = set(row.pred_types)
      j_score = len(true_set.intersection(pred_set)) / len(true_set.union(pred_set))
      scores.append(j_score)

  return np.mean(scores)

def binary_evaluate(true_label, pred_label, pos_label=1, neg_label=0):
    metrics = {
        'Accuracy':           accuracy_score(true_label, pred_label),
        'Macro Precision':    precision_score(true_label, pred_label, average='macro'),
        'Micro Precision':    precision_score(true_label, pred_label, average='micro'),
        'Macro Recall':       recall_score(true_label, pred_label, average='macro'),
        'Micro Recall':       recall_score(true_label, pred_label, average='micro'),
        'Macro F1':           f1_score(true_label, pred_label, average='macro'),
        'Micro F1':           f1_score(true_label, pred_label, average='micro'),
        'Binary F1 for yes class':           f1_score(true_label, pred_label, average='binary', pos_label=pos_label),
        'Binary F1 for no class':           f1_score(true_label, pred_label, average='binary', pos_label=neg_label),
        'Weighted F1':           f1_score(true_label, pred_label, average='weighted'),
        # 'Balanced Accuracy':  balanced_accuracy_score(true_label, pred_label),
    }

    keys_list = list(metrics.keys())   # preserves insertion order (Python 3.7+)
    return metrics, keys_list


def evaluate(true_label, pred_label, df_list):
    metrics = {
        'Accuracy':           accuracy_score(true_label, pred_label),
        'Macro Precision':    precision_score(true_label, pred_label, average='macro'),
        'Micro Precision':    precision_score(true_label, pred_label, average='micro'),
        'Macro Recall':       recall_score(true_label, pred_label, average='macro'),
        'Micro Recall':       recall_score(true_label, pred_label, average='micro'),
        'Macro F1':           f1_score(true_label, pred_label, average='macro'),
        'Micro F1':           f1_score(true_label, pred_label, average='micro'),
        'Jaccard Score': jaccard_score(df_list),
    }

    keys_list = list(metrics.keys())   # preserves insertion order (Python 3.7+)
    return metrics, keys_list


def average_hamming_dist(true_label, pred_label):
  dists = []
  for i in range(len(true_label)):
    dists.append(np.sum(true_label[i] != pred_label[i]))
  return np.mean(dists)


In [5]:
test_model = 'GPT'
models = ['baseline', 'COT', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']

all_results = dict()

for m in models:
  df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/GPT/{test_model}_{m}_test_data_pred.json')
  res, _ = binary_evaluate(df['label'], df['pred_label'], pos_label='yes stigma', neg_label='no stigma')

  for k in res:
    if k not in all_results:
      all_results[k] = []

    all_results[k].append(res[k])

res_df = pd.DataFrame(all_results, index=models)
res_df.to_csv(f'{test_model}_labels_results.csv')

In [25]:
from sklearn.preprocessing import MultiLabelBinarizer


test_model = 'Llama'
models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
classes=['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

all_results = dict()

for m in models:
  # print(m)
  df_label = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/{test_model}/{test_model}_label_{m}_test_data_pred.json')
  df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/{test_model}/{test_model}_type_{m}_test_data_pred.json')

  condition_idx = df_label[df_label['pred_label'] == 'yes stigma'].index
  f_df = df.loc[condition_idx]

  print(m, '=', len(f_df))

  mlb = MultiLabelBinarizer(classes=classes)
  label_1_hot = mlb.fit_transform(f_df['types'])
  pred_label_1_hot = mlb.transform(f_df['pred_types'])
  # print(f_df)

  res, _ = evaluate(label_1_hot, pred_label_1_hot, f_df)

  for k in res:
    if k not in all_results:
      all_results[k] = []

    all_results[k].append(res[k])

res_df = pd.DataFrame(all_results, index=models)
res_df.to_csv(f'{test_model}_types_results.csv')

baseline = 41
MIPROV2 = 122
bootstrap_fewshot = 122
bootstrap_fewshot_random_search = 122
labeled_fewshot = 52


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

bootstrap_fewshot_optuna = 11
COPRO = 41


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_

In [29]:
from sklearn.preprocessing import MultiLabelBinarizer


test_model = 'GPT'
models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
classes=['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

all_results = dict()

for m in models:
  # print(m)
  df_label = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/{test_model}/{test_model}_{m}_test_data_pred.json')
  df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/{test_model}/{test_model}_{m}_test_data_pred.json')

  condition_idx = df_label[df_label['label'] == 'yes stigma'].index
  f_df = df.loc[condition_idx]

  print(m, '=', len(f_df))

  mlb = MultiLabelBinarizer(classes=classes)
  label_1_hot = mlb.fit_transform(f_df['types'])
  pred_label_1_hot = mlb.transform(f_df['pred_types'])
  # print(f_df)

  res, _ = evaluate(label_1_hot, pred_label_1_hot, f_df)

  for k in res:
    if k not in all_results:
      all_results[k] = []

    all_results[k].append(res[k])

res_df = pd.DataFrame(all_results, index=models)
res_df.to_csv(f'{test_model}_types_label_filtered_results.csv')

baseline = 26
MIPROV2 = 26
bootstrap_fewshot = 26
bootstrap_fewshot_random_search = 26
labeled_fewshot = 26
bootstrap_fewshot_optuna = 26


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.p

COPRO = 26


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Hamming code

In [9]:
from sklearn.preprocessing import MultiLabelBinarizer


test_model = 'Llama'
models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
classes=['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

all_results = dict()
all_results['hamming'] = []

for m in models:
  # print(m)
  df_label = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy label optimization - pos samples/{test_model}/{test_model}_label_{m}_test_data_pred.json')
  # gpt_df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/GPT/GPT_{m}_test_data_pred.json')
  model_df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/DSPy type optimization - pos samples/{test_model}/{test_model}_type_{m}_test_data_pred.json')

  condition_idx = df_label[df_label['pred_label'] == 'yes stigma'].index
  model_f_df = model_df.loc[condition_idx]
  # llama_f_df = llama_df.loc[condition_idx]



  mlb = MultiLabelBinarizer(classes=classes)
  label_1_hot = mlb.fit_transform(model_f_df['types'])
  pred_label_1_hot = mlb.transform(model_f_df['pred_types'])
  # print(f_df)

  ans = average_hamming_dist(label_1_hot, pred_label_1_hot)
  all_results['hamming'].append(ans)

res_df = pd.DataFrame(all_results, index=models)
res_df.to_csv(f'{test_model}_avg_hamming_true_label_results.csv')

# Class wise

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer


test_model = 'Llama'
models = ['baseline', 'MIPROV2', 'bootstrap_fewshot', 'bootstrap_fewshot_random_search', 'labeled_fewshot', 'bootstrap_fewshot_optuna', 'COPRO']
classes=['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']


for i in range(len(classes)):
  all_results = dict()
  for m in models:
    df = pd.read_json(f'/content/drive/MyDrive/stigma_project/final experiment results/{test_model}_{m}_test_data_pred.json')
    mlb = MultiLabelBinarizer(classes=classes)
    label_1_hot = mlb.fit_transform(df['types'])
    pred_label_1_hot = mlb.transform(df['pred_types'])

    res, _ = binary_evaluate(label_1_hot[:, i], pred_label_1_hot[:, i])

    for k in res:
      if k not in all_results:
        all_results[k] = []

      all_results[k].append(res[k])

  res_df = pd.DataFrame(all_results, index=models)
  res_df.to_csv(f'/content/drive/MyDrive/stigma_project/final experiment results/result metrics/{test_model}_type {classes[i]}_results.csv')

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr